# Debugging & Reading Tracebacks
**Topic:** Python Fundamentals

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#read from the bottom up! u will see patterns

---
## What you'll explore

By the end of this demo you will be able to:

- **Read** a Python traceback from bottom to top and identify the line that actually caused the error
- **Explain** the difference between the error location and the error origin in a multi-step call stack
- **Interpret** common pandas and scikit-learn error messages and map them to the underlying cause

---
## How we got here

In *09: Error Handling* we learned to catch exceptions so pipelines do not crash. But first you have to understand what went wrong. A traceback is Python's way of telling you exactly where and why an exception occurred, and reading one efficiently is one of the most practical skills a data scientist can develop. This notebook walks through real broken data science code and shows you how to decode the output.

---
## Why this matters for data science

Pandas and scikit-learn tracebacks can be intimidating: they are often 15-30 lines long because the error propagates through several layers of library code before surfacing. Knowing that you should almost always read the **last two lines first** and then look at the **first occurrence of your own file** in the traceback cuts debugging time dramatically. This skill also transfers directly to reading GitHub issues and Stack Overflow answers for library bugs.

---
## Try it yourself

In [ ]:
# ▶ Run this cell and observe the output.
# Then try changing the values and running again.

# Intentional bug: what happens when you divide by zero?
def average(values):
    total = sum(values)
    count = len(values)
    return total / count

print(average([10, 20, 30]))   # works
# print(average([]))           # uncomment this line to trigger a ZeroDivisionError

In [ ]:
# ✏️ Your turn — modify this code:
# 1. Call lookup_grade(95) — what error do you get? Read the traceback carefully.
# 2. Add 95 to the grades dict and run again
# 3. What is the difference between a KeyError and an IndexError?

grades = {60: 'D', 70: 'C', 80: 'B', 90: 'A'}

def lookup_grade(score):
    bucket = (score // 10) * 10
    return grades[bucket]

print(lookup_grade(80))
print(lookup_grade(70))

In [ ]:
# 🎯 Challenge:
# This function has 3 bugs. Find and fix all of them:
#
# def calculate_average(numbers):
#     total = 0
#     for num in numbers        # bug 1
#         total += num
#     return total / len(numbers)
#     print(f'Average: {average}')  # bug 2 and bug 3
#
# Hint: look carefully at syntax, unreachable code, and undefined variable names

# Your corrected version here:

---
## What's happening?

A traceback is a record of the call stack at the moment an exception was raised. Python prints it with the most recent call at the **bottom** and the outermost call at the **top**. The last line is always the exception type and message, start there.

```
Traceback (most recent call last):
  File "pipeline.py", line 42, in run_pipeline     # <- your code
    df = load_and_clean(path)
  File "pipeline.py", line 18, in load_and_clean   # <- still your code
    df["salary"] = df["salary"].astype(float)
  File ".../pandas/core/series.py", line 400 ...   # <- pandas internals
    ...
ValueError: could not convert string to float: '$72,000'
```

| Reading strategy | What to do | Why it helps |
|-----------------|-----------|-------------|
| Start at the bottom | Read the exception type and message first | Tells you *what* went wrong |
| Find your code | Scan upward for lines from your own file | Tells you *where in your code* it went wrong |
| Read the error message literally | `"could not convert string to float: '$72,000'"` | Often contains the exact bad value |
| Check one level up | The line that called the failing function | Tells you *why* your code reached that state |

### Common pandas and scikit-learn error messages decoded

`ValueError: Input contains NaN` from sklearn means you passed missing values to a transformer that does not handle them. Impute or drop first.

`KeyError: 'column_name'` from pandas usually means a column was renamed, dropped, or never created upstream in the pipeline.

Return to the widget and step through all four scenarios to build pattern recognition for the most common tracebacks you will encounter.

---
## A direct example: the simplest possible call stack

One function calls another, the inner one fails. The traceback has three entries. The chart below represents those three frames — the same structure you will see in every traceback, just with more layers added.

- **Notice:** Start at the bottom (blue): it tells you what failed and the exact bad value
- **Notice:** The middle entry (dark red) tells you which line in your code triggered it
- **Notice:** The top entry (red) tells you what called that line — the context that explains why your code reached that state

In [ ]:
frames = [
    ("process_scores(raw)",          "your code — outer call",  "#E45756"),
    ("float('N/A')",                 "your code — inner call",  "#C0392B"),
    ("ValueError: could not convert string to float: 'N/A'",
                                     "Python raises here",      "#4C72B0"),
]

labels = [f[0] for f in frames]
annots = [f[1] for f in frames]
colors = [f[2] for f in frames]

fig, ax = plt.subplots(figsize=(9, 3))
bars = ax.barh(labels, [1] * len(frames), color=colors)
for bar, annot in zip(bars, annots):
    ax.text(0.5, bar.get_y() + bar.get_height() / 2, annot,
            ha="center", va="center", color="white", fontsize=11, fontweight="bold")
ax.set_title(
    "Read bottom to top: What failed → Where in your code → Why it was called",
    fontsize=12)
ax.set_xlim(0, 1.3)
ax.set_xticks([])
ax.invert_yaxis()
ax.spines[["top", "right", "bottom"]].set_visible(False)
plt.tight_layout()
plt.show()

---
## Real-world example: A broken feature engineering pipeline

The code below intentionally reproduces a common multi-step bug: a column is renamed in one step, then accessed by its old name in the next step. The chart visualizes the call stack depth at the moment of the crash.

Notice:

- **Notice:** The error fires 4 function calls deep into the pipeline, but the root cause is on line 2 of the user's code, where the column was renamed without updating downstream references
- **Notice:** The pandas internals account for most of the traceback frames, even though pandas itself did nothing wrong; this is normal and expected
- **Notice:** The fix is one character: changing `"salary"` to `"annual_salary"` in the downstream reference, which is why reading the error message for the exact bad key is faster than re-reading all the code

> **Discussion question:** A teammate suggests always using `df.get("salary", default=0)` instead of `df["salary"]` to avoid `KeyError`. What problem does this solve, and what new problem does it introduce?

In [ ]:
# ── Call stack depth visualization for a broken pipeline ─────────────────────
frames = [
    ("run_experiment()",       "Your code",     "#E45756"),
    ("run_pipeline(df)",       "Your code",     "#E45756"),
    ("build_features(df)",     "Your code",     "#E45756"),
    ("df['salary'].astype()",  "Your code",     "#E45756"),
    ("Series.__getitem__()",   "pandas",        "#8172B2"),
    ("Index.get_loc()",        "pandas",        "#8172B2"),
    ("raise KeyError(key)",    "pandas raises", "#4C72B0"),
]

labels = [f[0] for f in frames]
colors = [f[2] for f in frames]
annots = [f[1] for f in frames]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels, [1] * len(labels), color=colors)
for bar, annot in zip(bars, annots):
    ax.text(0.5, bar.get_y() + bar.get_height() / 2, annot,
            ha="center", va="center", color="white", fontsize=11, fontweight="bold")
ax.set_title(
    "Call Stack at Crash — KeyError in Feature Engineering Pipeline\n"
    "Red = your code  |  Purple = pandas internals  |  Blue = where error raised",
    fontsize=12)
ax.set_xlim(0, 1.3)
ax.set_xticks([])
ax.invert_yaxis()
ax.spines[["top", "right", "bottom"]].set_visible(False)
plt.tight_layout()
plt.show()


### Debugging strategy checklist

| Step | Action | Tool |
|------|--------|------|
| 1 | Read the last line of the traceback first | Terminal / Jupyter cell output |
| 2 | Identify the exception type and the exact bad value | Last line of traceback |
| 3 | Find the first frame in your own file | Scan upward from the bottom |
| 4 | Print or inspect the value at that line | `print()`, `df.head()`, `type()` |
| 5 | Check what the data looked like one step earlier | Add a `print()` or `display()` before the failing line |
| 6 | Search the error message verbatim | Stack Overflow / GitHub Issues |

---
## Key takeaway

> **Read a traceback bottom to top: the last line names the error, the first frame in your own code names the place, and the line above that names why — fixing bugs is mostly just reading carefully.**

---
*Next up: Virtual Environments & Packages — how to create isolated Python environments so your data science projects never conflict with each other*